# **PocketDoc - AI Powered Triage Assistant**
Your AI Powered second opinion, anytime.

CELL 1 - Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


CELL 2 - INSTALL LIBRARIES


In [ ]:
# CELL 2 - INSTALL LIBRARIES
!pip install sentence-transformers faiss-cpu groq datasets spacy -q
!python -m spacy download en_core_web_sm
print("Libraries installed — restart runtime now!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 75.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Libraries installed — restart runtime now!


CELL 3 — Load spaCy

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")
print("✅ spaCy ready!")

✅ spaCy ready!


CELL 4 - GROQ API

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # Load from .env file
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
print("Groq API key loaded!")

Groq API key set!


In [ ]:
from groq import Groq

client = Groq()

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Say hello in one word"}
    ]
)

print(response.choices[0].message.content)

Hello


CELL 5 — Reload Saved Data from Drive

In [ ]:
import numpy as np
import json
import faiss

embeddings = np.load('/content/drive/MyDrive/cliniq/embeddings/embeddings.npy')

with open('/content/drive/MyDrive/cliniq/embeddings/chunks.json', 'r') as f:
    all_chunks = json.load(f)

index = faiss.read_index('/content/drive/MyDrive/cliniq/embeddings/faiss_index.index')

print(f"Embeddings loaded: {embeddings.shape}")
print(f"Chunks loaded: {len(all_chunks)}")
print(f"FAISS index loaded: {index.ntotal} vectors")

Embeddings loaded: (30412, 384)
Chunks loaded: 30412
FAISS index loaded: 30412 vectors


UPLODE DATASET

In [ ]:
from datasets import load_dataset
print("Downloding from HuggingFace.")

try:
  dataset = load_dataset("lavita/MedQuAD", split="train")
  print(f"Dataset loded. Total rows: {len(dataset)}")
  print(f"Column names: {dataset.column_names}")
except Exception as e:
  print(f"Failed: {e}")

Downloding from HuggingFace.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-e36383d177026d(…):   0%|          | 0.00/10.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/47441 [00:00<?, ? examples/s]

Dataset loded. Total rows: 47441
Column names: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer']


In [ ]:
#save to drive
save_path = '/content/drive/MyDrive/cliniq/data/raw/medquad.csv'
df.to_csv(save_path, index=False)
print(f"Dataset saved to Drive! Shape: {df.shape}")


Dataset saved to Drive! Shape: (47441, 13)


CHUNKING & EMBEDDING


In [ ]:
def chunk_text(text, chunk_size=200, overlap=50):
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

# Apply chunking to all answers
all_chunks = []
for answer in df['answer']:
    if isinstance(answer, str) and len(answer.strip()) > 0:
        chunks = chunk_text(answer)
        all_chunks.extend(chunks)

print(f"Total chunks created: {len(all_chunks)}")
print(f"\nExample chunk:\n{all_chunks[0]}")

Total chunks created: 30412

Example chunk:
Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the soles of the feet to become thick, scaly, and calloused. Cardiomyopathy, which is a disease of the heart muscle, is a life-threatening health problem that can develop in people with keratoderma with woolly hair. Unlike the other features of this condition, signs and symptoms of cardiomyopathy may not appear until adolescence or later. Complications of cardiomyopathy can include an abnormal heartbeat (arrhythm

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(all_chunks, show_progress_bar=True, batch_size=64)
print(f"\n Embeddings created!")
print(f"Shape: {embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/476 [00:00<?, ?it/s]


 Embeddings created!
Shape: (30412, 384)


In [ ]:
import numpy as np

# Save embeddings
np.save('/content/drive/MyDrive/cliniq/embeddings/embeddings.npy', embeddings)

# Save chunks
import json
with open('/content/drive/MyDrive/cliniq/embeddings/chunks.json', 'w') as f:
    json.dump(all_chunks, f)

print("Embeddings saved!")
print("Chunks saved!")

Embeddings saved!
Chunks saved!


FAISS Index

In [ ]:
import faiss
import numpy as np

#convert embeddigs into float32
vectors = np.array(embeddings).astype('float32')

#dimension
#The dimension is the number of numbers in each vector (384).
dim = vectors.shape[1]

#Building the index
index = faiss.IndexFlatL2(dim)
index.add(vectors)

print(f"Total vectors in index: {index.ntotal}")

Total vectors in index: 30412


In [ ]:
faiss.write_index(index, '/content/drive/MyDrive/cliniq/embeddings/faiss_index.index')
print("FAISS index saved to Drive!")

FAISS index saved to Drive!


CELL 6 - NLP ENTITY EXTRACTION

In [ ]:
def extract_entities(text):
  response = client.chat.completions.create(
      model ="llama-3.3-70b-versatile",
      messages=[
          {"role": "user", "content": f"Extract symptoms, body parts and duration from this text: {text}. Return only this format:\nSymptoms: ...\nBody parts: ...\nDuration: ..."}]
  )
  return response.choices[0].message.content
print(extract_entities("I have chest pain and fever for 3 days"))

Symptoms: chest pain, fever
Body parts: chest
Duration: 3 days


CELL 7 - RAG PIPELINE

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded!")

def embed_query(text):
    return model.encode([text])

def search_chunks(query):
    query_embedding = embed_query(query)
    distances, indices = index.search(query_embedding, 5)
    results = [all_chunks[i] for i in indices[0]]
    return results

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


CELL 8 - ADAPTIVE TRIAGE PIPELINE


In [38]:
# Function 1 — Classify initial severity
def classify_severity(symptoms):
  try:

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "user", "content": f"Classify these symptoms as exactly one of: CRITICAL, MODERATE, MILD. Return only that one word, nothing else. Symptoms: {symptoms}"}
        ]
    )
    return response.choices[0].message.content.strip()
  except Exception as e:
        print('Error classifying severity. Defaulting to MODERATE.')
        return 'MODERATE'


# Function 2 — Re-evaluate severity with full context
def re_evaluate_severity(symptoms, answers):
  try:

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "user", "content": f"Original symptoms: {symptoms}\nPatient answers: {answers}\nBased on ALL this information, classify severity as CRITICAL, MODERATE, or MILD.\nReturn only one word."}
        ]
    )
    return response.choices[0].message.content.strip()
  except Exception as e:
        print('Error classifying severity. Defaulting to MODERATE.')
        return 'MODERATE'


# Function 3 — Adaptive questioning
def adaptive_questions(symptoms, severity):
    answers = []

    limits = {"CRITICAL": 4, "MODERATE": 3, "MILD": 2}
    max_questions = limits.get(severity, 3)

    for i in range(max_questions):
      try:

        conversation = f"Symptoms: {symptoms}\nSeverity: {severity}\nAnswers so far: {answers}"

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "user", "content": f"You are a triage nurse.\n{conversation}\nAsk only about:\n- Pain characteristics (location, intensity, type)\n- Associated symptoms\n- Duration and progression\n- Relevant medical history\nAsk ONE question or say ENOUGH_INFO if you have enough.\nReturn only the question or ENOUGH_INFO."}
            ]
        )

        question = response.choices[0].message.content.strip()

        if "ENOUGH_INFO" in question:
            break

        print(f"\n🏥 PocketDoc: {question}")
        answer = input("Your answer: ")
        answers.append(f"{question} A: {answer}")

        # Re-evaluate severity with full context
        new_severity = re_evaluate_severity(symptoms, answers)
        if new_severity != severity:
            print(f"\nRisk updated: {severity} → {new_severity}")
            severity = new_severity
            max_questions = limits.get(severity, 3)

      except Exception as e:
        print('Error classifying severity. Defaulting to MODERATE.')
        return 'MODERATE'


    return answers, severity


# Function 4 — Generate final structured report
def generate_final_report(symptoms, severity, answers, chunks):
  try:

    context = "\n\n".join(chunks)
    answers_text = "\n".join(answers)

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "user", "content": f"""You are a clinical triage assistant.

Medical References: {context}

Patient Symptoms: {symptoms}
Severity: {severity}
Follow up answers: {answers_text}

Generate a structured clinical report with these exact sections:

1. SYMPTOMS SUMMARY
2. RED FLAGS
3. TRIAGE LEVEL: {severity}
4. POSSIBLE CONDITIONS
5. RECOMMENDED TESTS
6. FOLLOW UP CARE
"""}
        ]
    )
    return response.choices[0].message.content
  except Exception as e:
    print('Error classifying severity. Defaulting to MODERATE.')
    return 'MODERATE'


CELL 9 - EVALUATION METRICS

In [ ]:
import time

# TEST 1 — Severity Consistency

print("TEST 1 — Severity Consistency")

results = []
for i in range(3):
    result = classify_severity("chest pain and fever for 3 days")
    results.append(result)
    print(f"Run {i+1}: {result}")

if len(set(results)) == 1:
    print("Consistent\n")
else:
    print("Inconsistent\n")



# TEST 2 — Retrieval Relevance

print("TEST 2 — Retrieval Relevance")

query = "chest pain"
chunks = search_chunks(query)
keywords = ["chest", "pain", "cardiac", "heart", "pericarditis", "breathing"]
relevant = 0
for i, chunk in enumerate(chunks):
    chunk_lower = chunk.lower()
    matches = [k for k in keywords if k in chunk_lower]
    if matches:
        relevant += 1
        print(f"Chunk {i+1}: relevant — matched {matches}")
    else:
        print(f"Chunk {i+1}: not relevant")
print(f"\nRelevance Score: {relevant}/5\n")


# TEST 3 — Response Time

print("TEST 3 — Response Time")

start = time.time()
chunks = search_chunks("chest pain and fever")
generate_final_report("chest pain and fever", "CRITICAL", [], chunks)
end = time.time()
print(f"Full pipeline response time: {round(end - start, 2)} seconds")


TEST 1 — Severity Consistency
Run 1: CRITICAL
Run 2: CRITICAL
Run 3: CRITICAL
Consistent

TEST 2 — Retrieval Relevance
Chunk 1: relevant — matched ['chest', 'pain']
Chunk 2: relevant — matched ['chest', 'pain', 'heart']
Chunk 3: relevant — matched ['chest', 'pain', 'heart']
Chunk 4: relevant — matched ['chest', 'pain', 'heart']
Chunk 5: relevant — matched ['chest', 'pain', 'heart', 'pericarditis']

Relevance Score: 5/5

TEST 3 — Response Time
Full pipeline response time: 1.74 seconds


CELL 10 - POCKETDOC FINAL PIPELINE

In [39]:
def pocketdoc():

    print("Welcome to PocketDoc")
    print("Your AI powered second opinion, anytime.")
    print("Fast. Clear. Reliable.")

    try:

      # Step 1 — get symptoms from user

      symptoms = input('\nDescribe your symptoms: ')

      # Validate input is medical
      validation = client.chat.completions.create(
      model='llama-3.3-70b-versatile',
      messages=[
          {'role': 'user', 'content': f'Is this a medical symptom or health complaint? Answer only YES or NO. Input: {symptoms}'}
      ]
      )

      is_medical = validation.choices[0].message.content.strip().upper()

      if 'NO' in is_medical:
        print('\n⚠️ PocketDoc is designed for medical symptoms only.')
        print('Please describe a health concern or physical symptom.')
        return


      # Step 2 — classify severity
      print("\nAnalyzing symptoms...")
      severity = classify_severity(symptoms)
      print(f"Initial severity: {severity}")

      # Step 3 — adaptive questions
      print("\nLet me ask you a few questions...")
      answers, final_severity = adaptive_questions(symptoms, severity)

      # Step 4 — search chunks
      print("\nSearching medical database...")
      chunks = search_chunks(symptoms)

      # Step 5 — generate report
      print("\nGenerating your report...")
      report = generate_final_report(symptoms, final_severity, answers, chunks)

      # Step 6 — print report

      print("YOUR POCKETDOC REPORT")
      if final_severity == "CRITICAL":
          badge = "CRITICAL — Seek emergency care immediately"
      elif final_severity == "MODERATE":
          badge = "MODERATE — See a doctor within 24-48 hours"
      else:
          badge = "MILD — Monitor at home, rest and hydrate"

      print(f"\nRISK LEVEL: {badge}\n")
      print(report)
      print("\nDisclaimer: This is not a substitute for professional medical advice.")
    except Exception as e:
      print('\n⚠️ Something went wrong. Please restart and try again.')



In [ ]:
pocketdoc()

Welcome to PocketDoc
Your AI powered second opinion, anytime.
Fast. Clear. Reliable.

Describe your symptoms: cold and fever from 4 days

Analyzing symptoms...
Initial severity: MODERATE

Let me ask you a few questions...

🏥 PocketDoc: What type of pain are you experiencing, if any, and where is it located, such as your head, throat, or body aches?
Your answer: full body pains, throat pain wile swallowing and ears pain too

🏥 PocketDoc: On a scale of 1 to 10, how would you rate the intensity of your full body pains, and have you noticed any changes in the severity of your throat and ear pains over the past 4 days?
Your answer: 8

🏥 PocketDoc: Have you experienced any worsening or improvement in your symptoms, particularly the full body pains, throat pain, and ear pain, over the past 24 hours, and are there any other associated symptoms such as headache, soreness, or difficulty breathing?
Your answer: yes my temperature rased from 99 to 102 in 2 days 

Searching medical database...

Gen

In [ ]:
# Save notebook as Python file
from google.colab import drive

# Just save the key functions directly
with open('/content/drive/MyDrive/cliniq/src/pocketdoc.py', 'w') as f:
    import inspect
    f.write(inspect.getsource(classify_severity) + '\n\n')
    f.write(inspect.getsource(re_evaluate_severity) + '\n\n')
    f.write(inspect.getsource(adaptive_questions) + '\n\n')
    f.write(inspect.getsource(generate_final_report) + '\n\n')
    f.write(inspect.getsource(search_chunks) + '\n\n')
    f.write(inspect.getsource(embed_query) + '\n\n')
    f.write(inspect.getsource(pocketdoc) + '\n\n')

print("All functions saved to Drive!")

All functions saved to Drive!


In [ ]:
pocketdoc()


Welcome to PocketDoc
Your AI powered second opinion, anytime.
Fast. Clear. Reliable.

Describe your symptoms: fever from 2 days

Analyzing symptoms...
Initial severity: MODERATE

Let me ask you a few questions...

🏥 PocketDoc: Can you describe the location and intensity of any pain you're experiencing, and have you noticed any changes in the pain over the past 2 days since the fever started?
Your answer: body pains 

Risk updated: MODERATE → MILD

🏥 PocketDoc: Are the body pains you're experiencing constant or do they come and go, and have you noticed any areas of the body where the pain is more severe than others?
Your answer: they are constant 

Risk updated: MILD → MODERATE

🏥 PocketDoc: Are there any areas of the body where the constant body pains are more severe, such as the head, muscles, or joints, and are the pains sharp, dull, or aching in nature?
Your answer: no

Risk updated: MODERATE → MILD

Searching medical database...

Generating your report...
YOUR POCKETDOC REPORT

RIS

In [ ]:
pocketdoc()


Welcome to PocketDoc
Your AI powered second opinion, anytime.
Fast. Clear. Reliable.

Describe your symptoms: i have chest pain and fever since 3 days.

Analyzing symptoms...
Initial severity: CRITICAL

Let me ask you a few questions...

🏥 PocketDoc: What is the nature and intensity of your chest pain, is it a sharp, dull, burning, or squeezing sensation, and does it radiate to any other parts of your body?
Your answer: its paining when i try to breath.

🏥 PocketDoc: Have you experienced any other symptoms such as cough, shortness of breath, or fatigue alongside the chest pain and fever, and has the intensity of the chest pain increased or decreased over the past 3 days?
Your answer: yes sometimes

Risk updated: CRITICAL → MODERATE

🏥 PocketDoc: Have you had any previous conditions or surgeries related to your heart or lungs that could be contributing to your current symptoms of chest pain and fever?
Your answer: no

🏥 PocketDoc: Have you noticed any changes in the intensity or charact

In [26]:
pocketdoc()


Welcome to PocketDoc
Your AI powered second opinion, anytime.
Fast. Clear. Reliable.

Describe your symptoms: i love formula one

Analyzing symptoms...
Initial severity: MILD

Let me ask you a few questions...

🏥 PocketDoc: Can you tell me if you're experiencing any physical symptoms, such as pain or discomfort, while watching or thinking about Formula One, and if so, where is it located and how would you describe it?
Your answer: no

🏥 PocketDoc: Have you noticed any increase or change in your enthusiasm or excitement while watching Formula One that is causing you any emotional or psychological distress, such as anxiety or euphoria, and if so, how long have you been experiencing this?
Your answer: no

Searching medical database...

Generating your report...
YOUR POCKETDOC REPORT

RISK LEVEL: MILD — Monitor at home, rest and hydrate

**1. SYMPTOMS SUMMARY**
The patient reports enjoying Formula One, with no associated physical symptoms such as pain or discomfort. The patient also denies

In [34]:
pocketdoc()

Welcome to PocketDoc
Your AI powered second opinion, anytime.
Fast. Clear. Reliable.

Describe your symptoms: i hate hospital

⚠️ PocketDoc is designed for medical symptoms only.
Please describe a health concern or physical symptom.
